# Engine PINN Standalone Notebook (Physics-Updated)

Self-contained notebook. You only need:
1) this notebook file
2) optional CSV with columns: `BMEP`, `H2_percentage`, `Spark_Ignition_Timing`, `Lambda`, `BSFC`, `NOx` (+ optional `LHV`)

If no CSV is found, realistic synthetic data is generated.


In [ ]:
import importlib
import subprocess
import sys

required = ["numpy", "pandas", "torch", "sklearn", "matplotlib", "seaborn"]
missing = []
for pkg in required:
    try:
        importlib.import_module(pkg)
    except Exception:
        missing.append(pkg)

if missing:
    print("Installing:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("Dependencies already installed.")


In [ ]:
import logging
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader, Dataset

sns.set(style="whitegrid")


In [ ]:
@dataclass
class PhysicsConfig:
    nox_b_init: float = 5.0

@dataclass
class TrainConfig:
    seed: int = 42
    batch_size: int = 128
    lr: float = 1e-3
    weight_decay: float = 1e-5
    lambda_phys: float = 10.0
    lambda_mono: float = 2.0
    epochs_phase1: int = 25
    epochs_phase2: int = 50
    patience: int = 15
    test_size: float = 0.15
    val_size: float = 0.15
    data_csv: Path = Path("engine_data.csv")
    artifacts_dir: Path = Path("artifacts")


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(name)s | %(message)s")
logger = logging.getLogger("engine-pinn-standalone")

train_cfg = TrainConfig()
phys_cfg = PhysicsConfig()
set_seed(train_cfg.seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


In [ ]:
FEATURE_COLUMNS = ["BMEP", "H2_percentage", "Spark_Ignition_Timing", "Lambda"]
TARGET_COLUMNS = ["BSFC", "NOx"]


def h2_to_lhv(h2_percentage: float) -> float:
    if h2_percentage < 9:
        return 49.93
    if h2_percentage < 21.5:
        return 52.39
    if h2_percentage < 27.5:
        return 53.59
    return 54.58


class EngineDataset(Dataset):
    def __init__(self, x_scaled, y_scaled, x_raw, y_raw, lhv_raw):
        if len(x_scaled) != len(y_scaled):
            raise ValueError("Mismatched lengths")
        self.x = torch.tensor(x_scaled, dtype=torch.float32)
        self.y = torch.tensor(y_scaled, dtype=torch.float32)
        self.x_raw = torch.tensor(x_raw, dtype=torch.float32)
        self.y_raw = torch.tensor(y_raw, dtype=torch.float32)
        self.lhv_raw = torch.tensor(lhv_raw.reshape(-1, 1), dtype=torch.float32)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return {
            "x": self.x[idx],
            "y": self.y[idx],
            "x_raw": self.x_raw[idx],
            "y_raw": self.y_raw[idx],
            "lhv_raw": self.lhv_raw[idx],
        }


def load_or_generate_dataframe(csv_path: Path, n_samples: int = 1500, random_state: int = 42):
    if csv_path.exists():
        df = pd.read_csv(csv_path)
        required = set(FEATURE_COLUMNS + TARGET_COLUMNS)
        missing = required.difference(df.columns)
        if missing:
            raise ValueError(f"CSV missing required columns: {sorted(missing)}")
        if "LHV" not in df.columns:
            df["LHV"] = df["H2_percentage"].apply(h2_to_lhv)
        return df

    rng = np.random.default_rng(random_state)
    bmep = rng.uniform(2.0, 18.0, n_samples)
    h2 = rng.uniform(0.0, 40.0, n_samples)
    spark = rng.uniform(-10.0, 30.0, n_samples)
    lamb = rng.uniform(0.85, 1.25, n_samples)
    lhv_values = np.array([h2_to_lhv(h) for h in h2])

    fmep = 0.4 + 0.08 * bmep + 0.02 * rng.normal(size=n_samples)
    eff = np.clip(0.28 + 0.015 * bmep - 0.0015 * (spark - 10) ** 2 + 0.003 * h2, 0.2, 0.8)
    temp = 800 + 9.0 * spark + 6.5 * bmep + 2.0 * h2 + 20 * rng.normal(size=n_samples)
    temp = np.clip(temp, 300, None)

    bsfc = (3600.0 / (lhv_values * 1000.0)) * ((bmep + fmep) / (np.maximum(bmep, 1e-3) * eff))
    bsfc *= 1.0 + 0.02 * rng.normal(size=n_samples)

    a_true, b_true, c_true = 1200.0, 4.8, 1.25
    l3 = np.maximum(temp / 1000.0, 1e-4)
    nox = a_true * np.power(l3, c_true) * np.exp(-b_true / l3)
    nox *= (1 + 0.08 * np.maximum(spark, 0) / 30.0)
    nox *= 1.0 + 0.03 * rng.normal(size=n_samples)
    nox = np.clip(nox, 1e-3, None)

    return pd.DataFrame({
        "BMEP": bmep,
        "H2_percentage": h2,
        "Spark_Ignition_Timing": spark,
        "Lambda": lamb,
        "BSFC": bsfc,
        "NOx": nox,
        "LHV": lhv_values,
    })


def build_dataloaders(df: pd.DataFrame, batch_size: int, test_size: float, val_size: float, seed: int):
    train_df, test_df = train_test_split(df, test_size=test_size, random_state=seed)
    val_ratio = val_size / (1 - test_size)
    train_df, val_df = train_test_split(train_df, test_size=val_ratio, random_state=seed)

    x_scaler, y_scaler = StandardScaler(), StandardScaler()

    x_train = x_scaler.fit_transform(train_df[FEATURE_COLUMNS])
    y_train = y_scaler.fit_transform(train_df[TARGET_COLUMNS])
    x_val = x_scaler.transform(val_df[FEATURE_COLUMNS])
    y_val = y_scaler.transform(val_df[TARGET_COLUMNS])
    x_test = x_scaler.transform(test_df[FEATURE_COLUMNS])
    y_test = y_scaler.transform(test_df[TARGET_COLUMNS])

    if "LHV" in train_df.columns:
        lhv_train = train_df["LHV"].to_numpy()
        lhv_val = val_df["LHV"].to_numpy()
        lhv_test = test_df["LHV"].to_numpy()
    else:
        lhv_train = np.array([h2_to_lhv(h) for h in train_df["H2_percentage"]])
        lhv_val = np.array([h2_to_lhv(h) for h in val_df["H2_percentage"]])
        lhv_test = np.array([h2_to_lhv(h) for h in test_df["H2_percentage"]])

    train_ds = EngineDataset(x_train, y_train, train_df[FEATURE_COLUMNS].to_numpy(), train_df[TARGET_COLUMNS].to_numpy(), lhv_train)
    val_ds = EngineDataset(x_val, y_val, val_df[FEATURE_COLUMNS].to_numpy(), val_df[TARGET_COLUMNS].to_numpy(), lhv_val)
    test_ds = EngineDataset(x_test, y_test, test_df[FEATURE_COLUMNS].to_numpy(), test_df[TARGET_COLUMNS].to_numpy(), lhv_test)

    return (
        DataLoader(train_ds, batch_size=batch_size, shuffle=True),
        DataLoader(val_ds, batch_size=batch_size, shuffle=False),
        DataLoader(test_ds, batch_size=batch_size, shuffle=False),
        x_scaler,
        y_scaler,
    )


In [ ]:
class EfficiencyActivation(nn.Module):
    def __init__(self, low: float = 0.2, high: float = 0.8):
        super().__init__()
        self.low = low
        self.scale = high - low

    def forward(self, x):
        return self.low + self.scale * torch.sigmoid(x)


class LatentActivationBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.softplus = nn.Softplus()
        self.eff = EfficiencyActivation(0.2, 0.8)

    def forward(self, z):
        l1 = self.softplus(z[:, 0:1])
        l2 = self.eff(z[:, 1:2])
        l3 = self.softplus(z[:, 2:3])
        return torch.cat([l1, l2, l3], dim=-1)


class EnginePINN(nn.Module):
    def __init__(self, input_dim: int = 4, hidden_dims: Tuple[int, int] = (64, 32), nox_a_init: float = 500.0, nox_b_init: float = 5.0):
        super().__init__()
        h1, h2 = hidden_dims
        self.feature_extractor = nn.Sequential(
            nn.Linear(input_dim, h1), nn.BatchNorm1d(h1), nn.ReLU(),
            nn.Linear(h1, h2), nn.BatchNorm1d(h2), nn.ReLU(),
        )
        self.latent_head = nn.Linear(h2, 3)
        self.latent_act = LatentActivationBlock()

        self.nox_a_raw = nn.Parameter(torch.tensor(float(max(nox_a_init, 1e-3))))
        self.nox_b_raw = nn.Parameter(torch.tensor(float(max(nox_b_init, 1e-3))))
        self.nox_c_raw = nn.Parameter(torch.tensor(1.0))

        nn.init.xavier_uniform_(self.latent_head.weight)
        with torch.no_grad():
            self.latent_head.bias[0] = 0.5
            l2_target = (0.35 - 0.2) / 0.6
            self.latent_head.bias[1] = torch.log(torch.tensor(l2_target / (1 - l2_target)))
            self.latent_head.bias[2] = 1.0

    @property
    def nox_a(self):
        return torch.nn.functional.softplus(self.nox_a_raw)

    @property
    def nox_b(self):
        return torch.nn.functional.softplus(self.nox_b_raw)

    @property
    def nox_c(self):
        return torch.nn.functional.softplus(self.nox_c_raw) + 0.5

    def set_physics_scalar_training(self, enabled: bool):
        self.nox_a_raw.requires_grad = enabled
        self.nox_b_raw.requires_grad = enabled
        self.nox_c_raw.requires_grad = enabled

    def forward(self, x_scaled, x_raw, lhv_raw):
        eps = 1e-6
        feats = self.feature_extractor(x_scaled)
        z = self.latent_act(self.latent_head(feats))
        bmep = x_raw[:, 0:1]
        l1, l2, l3 = z[:, 0:1], z[:, 1:2], z[:, 2:3]

        lhv_kj = torch.clamp(lhv_raw * 1000.0, min=eps)
        bsfc = (3600.0 / lhv_kj) * ((bmep + l1) / (torch.clamp(bmep, min=eps) * torch.clamp(l2, min=eps)))
        l3_safe = torch.clamp(l3, min=eps)
        nox = self.nox_a * torch.pow(l3_safe, self.nox_c) * torch.exp(-self.nox_b / l3_safe)
        nox = torch.clamp(nox, min=eps)

        return {"bsfc": bsfc, "nox": nox, "latents": z}


In [ ]:
class PINNLoss(nn.Module):
    def __init__(self, lambda_phys: float = 10.0, lambda_mono: float = 2.0):
        super().__init__()
        self.lambda_phys = lambda_phys
        self.lambda_mono = lambda_mono
        self.mse = nn.MSELoss()

    def _physics_penalty(self, z):
        l1, l2, l3 = z[:, 0], z[:, 1], z[:, 2]
        return (
            torch.relu(-l1).pow(2).mean()
            + torch.relu(0.2 - l2).pow(2).mean()
            + torch.relu(l2 - 1.0).pow(2).mean()
            + torch.relu(-l3).pow(2).mean()
        )

    def _mono_penalty(self, x_raw, out):
        bsfc_pred, nox_pred, latents = out["bsfc"], out["nox"], out["latents"]

        g_bsfc = torch.autograd.grad(bsfc_pred, x_raw, grad_outputs=torch.ones_like(bsfc_pred), create_graph=True, retain_graph=True, allow_unused=True)[0]
        g_nox = torch.autograd.grad(nox_pred, x_raw, grad_outputs=torch.ones_like(nox_pred), create_graph=True, retain_graph=True, allow_unused=True)[0]
        l1 = latents[:, 0:1]
        g_l1 = torch.autograd.grad(l1, x_raw, grad_outputs=torch.ones_like(l1), create_graph=True, retain_graph=True, allow_unused=True)[0]

        if g_bsfc is None:
            g_bsfc = torch.zeros_like(x_raw)
        if g_nox is None:
            g_nox = torch.zeros_like(x_raw)
        if g_l1 is None:
            g_l1 = torch.zeros_like(x_raw)

        dbsfc_dbmep = g_bsfc[:, 0]
        dnox_dbmep = g_nox[:, 0]
        dnox_dh2 = g_nox[:, 1]
        dnox_dspark = g_nox[:, 2]
        dl1_dbmep = g_l1[:, 0]

        mono = (
            torch.relu(dbsfc_dbmep).pow(2).mean()
            + torch.relu(-dnox_dbmep).pow(2).mean()
            + torch.relu(-dnox_dh2).pow(2).mean()
            + torch.relu(-dnox_dspark).pow(2).mean()
            + torch.relu(-dl1_dbmep).pow(2).mean()
        )
        return mono

    def forward(self, out, y_raw, x_raw):
        eps = 1e-6
        bsfc_true = y_raw[:, 0:1]
        nox_true = torch.clamp(y_raw[:, 1:2], min=eps)

        data = self.mse(out["bsfc"], bsfc_true) + self.mse(torch.log(torch.clamp(out["nox"], min=eps)), torch.log(nox_true))
        phys = self._physics_penalty(out["latents"])
        mono = self._mono_penalty(x_raw, out)
        total = data + self.lambda_phys * phys + self.lambda_mono * mono
        return total, data, phys, mono


class Trainer:
    def __init__(self, model, loss_fn, optimizer, device, checkpoint_dir: Path, patience: int = 15):
        self.model = model
        self.loss_fn = loss_fn
        self.optimizer = optimizer
        self.device = device
        self.checkpoint_dir = checkpoint_dir
        self.checkpoint_dir.mkdir(parents=True, exist_ok=True)
        self.scheduler = ReduceLROnPlateau(self.optimizer, mode="min", factor=0.5, patience=6)
        self.patience = patience

    def _step(self, batch, training: bool):
        x = batch["x"].to(self.device)
        y_raw = batch["y_raw"].to(self.device)
        x_raw = batch["x_raw"].to(self.device).detach().requires_grad_(True)
        lhv_raw = batch["lhv_raw"].to(self.device)

        if training:
            self.optimizer.zero_grad(set_to_none=True)

        out = self.model(x, x_raw, lhv_raw)
        total, data, phys, mono = self.loss_fn(out, y_raw, x_raw)

        if training:
            total.backward()
            self.optimizer.step()

        return float(total.detach().cpu()), float(data.detach().cpu()), float(phys.detach().cpu()), float(mono.detach().cpu())

    def _run_epoch(self, loader, training: bool):
        self.model.train(training)
        vals = [self._step(batch, training) for batch in loader]
        return np.mean(np.array(vals), axis=0)

    def fit(self, train_loader, val_loader, epochs_phase1: int, epochs_phase2: int):
        best_val = float("inf")
        no_improve = 0
        history = {"train_total": [], "val_total": []}

        total_epochs = epochs_phase1 + epochs_phase2
        for epoch in range(total_epochs):
            phase = 1 if epoch < epochs_phase1 else 2
            self.model.set_physics_scalar_training(enabled=(phase == 2))

            tr_total, tr_data, tr_phys, tr_mono = self._run_epoch(train_loader, training=True)
            va_total, va_data, va_phys, va_mono = self._run_epoch(val_loader, training=False)

            history["train_total"].append(float(tr_total))
            history["val_total"].append(float(va_total))
            self.scheduler.step(float(va_total))

            logger.info(
                "Epoch %d/%d phase=%d train=%.6f val=%.6f data=%.6f phys=%.6f mono=%.6f",
                epoch + 1, total_epochs, phase, tr_total, va_total, va_data, va_phys, va_mono,
            )

            if va_total < best_val:
                best_val = float(va_total)
                no_improve = 0
                torch.save(self.model.state_dict(), self.checkpoint_dir / "best_model.pt")
            else:
                no_improve += 1
                if no_improve >= self.patience:
                    logger.info("Early stopping at epoch %d", epoch + 1)
                    break

        ckpt = self.checkpoint_dir / "best_model.pt"
        if ckpt.exists():
            self.model.load_state_dict(torch.load(ckpt, map_location=self.device))

        return history


In [ ]:
def evaluate_and_collect_latents(model, loader, y_scaler, device):
    model.eval()
    rows = []
    with torch.no_grad():
        for batch in loader:
            x = batch["x"].to(device)
            x_raw = batch["x_raw"].to(device)
            lhv_raw = batch["lhv_raw"].to(device)
            out = model(x, x_raw, lhv_raw)

            y_true = y_scaler.inverse_transform(batch["y"].cpu().numpy())
            y_pred = np.hstack([out["bsfc"].cpu().numpy(), out["nox"].cpu().numpy()])
            latent = out["latents"].cpu().numpy()
            x_raw_np = batch["x_raw"].cpu().numpy()
            lhv_np = batch["lhv_raw"].cpu().numpy()

            for i in range(len(x_raw_np)):
                rows.append({
                    "BMEP": x_raw_np[i, 0],
                    "H2_percentage": x_raw_np[i, 1],
                    "Spark_Ignition_Timing": x_raw_np[i, 2],
                    "Lambda": x_raw_np[i, 3],
                    "LHV": lhv_np[i, 0],
                    "BSFC_true": y_true[i, 0],
                    "NOx_true": y_true[i, 1],
                    "BSFC_pred": y_pred[i, 0],
                    "NOx_pred": y_pred[i, 1],
                    "L1_FMEP": latent[i, 0],
                    "L2_Indicated_Efficiency": latent[i, 1],
                    "L3_Temp_Potential": latent[i, 2],
                })
    return pd.DataFrame(rows)


def plot_training_curves(history, save_path: Path):
    save_path.parent.mkdir(parents=True, exist_ok=True)
    plt.figure(figsize=(8, 4))
    plt.plot(history["train_total"], label="Train")
    plt.plot(history["val_total"], label="Validation")
    plt.xlabel("Epoch")
    plt.ylabel("Total Loss")
    plt.legend()
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()


def plot_latent_trends(df, save_path: Path):
    save_path.parent.mkdir(parents=True, exist_ok=True)
    latents = ["L1_FMEP", "L2_Indicated_Efficiency", "L3_Temp_Potential"]
    inputs = ["BMEP", "H2_percentage", "Spark_Ignition_Timing", "Lambda"]

    fig, axes = plt.subplots(3, 4, figsize=(16, 10), sharey="row")
    for r, lat in enumerate(latents):
        for c, inp in enumerate(inputs):
            sns.regplot(data=df, x=inp, y=lat, ax=axes[r, c], scatter_kws={"s": 10, "alpha": 0.35}, line_kws={"color": "red"})
            axes[r, c].set_ylabel(lat if c == 0 else "")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close(fig)


def plot_parity_and_residuals(df, artifacts_dir: Path):
    artifacts_dir.mkdir(parents=True, exist_ok=True)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for ax, target in zip(axes, ["BSFC", "NOx"]):
        true = df[f"{target}_true"]
        pred = df[f"{target}_pred"]
        ax.scatter(true, pred, alpha=0.3)
        mn, mx = min(true.min(), pred.min()), max(true.max(), pred.max())
        ax.plot([mn, mx], [mn, mx], "r--")
        r2 = r2_score(true, pred)
        mae = mean_absolute_error(true, pred)
        ax.set_xlabel(f"True {target}")
        ax.set_ylabel(f"Predicted {target}")
        ax.set_title(f"{target}: R²={r2:.3f}, MAE={mae:.3f}")
    plt.tight_layout()
    plt.savefig(artifacts_dir / "parity_plots.png", dpi=150)
    plt.close(fig)

    tmp = df.copy()
    tmp["BSFC_resid"] = tmp["BSFC_pred"] - tmp["BSFC_true"]
    tmp["NOx_resid"] = tmp["NOx_pred"] - tmp["NOx_true"]
    inputs = ["BMEP", "H2_percentage", "Spark_Ignition_Timing", "Lambda"]

    fig, axes = plt.subplots(2, 4, figsize=(16, 8), sharey="row")
    for i, inp in enumerate(inputs):
        sns.scatterplot(data=tmp, x=inp, y="BSFC_resid", ax=axes[0, i], s=12, alpha=0.4)
        axes[0, i].axhline(0, color="r", linestyle="--")
        sns.scatterplot(data=tmp, x=inp, y="NOx_resid", ax=axes[1, i], s=12, alpha=0.4)
        axes[1, i].axhline(0, color="r", linestyle="--")
    plt.tight_layout()
    plt.savefig(artifacts_dir / "residuals_vs_inputs.png", dpi=150)
    plt.close(fig)


In [ ]:
# Optional: set your own data path
# train_cfg.data_csv = Path('/content/engine_data.csv')

df = load_or_generate_dataframe(train_cfg.data_csv)
print("Rows:", len(df))
print(df.head())

train_loader, val_loader, test_loader, x_scaler, y_scaler = build_dataloaders(
    df,
    batch_size=train_cfg.batch_size,
    test_size=train_cfg.test_size,
    val_size=train_cfg.val_size,
    seed=train_cfg.seed,
)

nox_a_init = max(float(df["NOx"].max()), 1e-3)
model = EnginePINN(nox_a_init=nox_a_init, nox_b_init=phys_cfg.nox_b_init).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=train_cfg.lr, weight_decay=train_cfg.weight_decay)
loss_fn = PINNLoss(lambda_phys=train_cfg.lambda_phys, lambda_mono=train_cfg.lambda_mono)
trainer = Trainer(model, loss_fn, optimizer, device, train_cfg.artifacts_dir / "checkpoints", patience=train_cfg.patience)

history = trainer.fit(train_loader, val_loader, train_cfg.epochs_phase1, train_cfg.epochs_phase2)
results_df = evaluate_and_collect_latents(model, test_loader, y_scaler, device)

train_cfg.artifacts_dir.mkdir(parents=True, exist_ok=True)
plot_training_curves(history, train_cfg.artifacts_dir / "training_curves.png")
plot_latent_trends(results_df, train_cfg.artifacts_dir / "latent_sensitivity.png")
plot_parity_and_residuals(results_df, train_cfg.artifacts_dir)
results_df.to_csv(train_cfg.artifacts_dir / "test_predictions_with_latents.csv", index=False)

print("Artifacts saved to:", train_cfg.artifacts_dir)


In [ ]:
def mape(y_true, y_pred):
    eps = 1e-8
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return float(np.mean(np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), eps))) * 100.0)

for target in ["BSFC", "NOx"]:
    yt = results_df[f"{target}_true"].to_numpy()
    yp = results_df[f"{target}_pred"].to_numpy()
    rmse = np.sqrt(mean_squared_error(yt, yp))
    r2 = r2_score(yt, yp)
    mae = mean_absolute_error(yt, yp)
    print(f"{target}: RMSE={rmse:.6f} | R2={r2:.4f} | MAE={mae:.6f} | MAPE={mape(yt, yp):.2f}%")

results_df.head()
